# Convex Cost Structure vs RL Pricing Accuracy
This notebook explores how the convex cost parameters (`c` and `gamma`) relate to the percent difference between LSM and RL swing option prices.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

In [2]:
data_path = "Convex Costs Results 6.csv"
df = pd.read_csv(data_path)
df.head()

,Configuration,c,gamma,Best Seed,Best Episode,LSM Price,RL Price,PctDiff,CI_95,MeanDiff,CI_Lower,CI_Upper,RL_95CI,LSM_95CI,PctDiff_95CI
0,SwingOption_20_c0.00_gamma1,0.00,1.0,11,23552,2.6612,2.6585,-0.10,NaN,NaN,NaN,NaN,0.019920,0.021322,0.102916
1,SwingOption_20_c0.01_gamma1,0.01,1.0,12,29696,2.5478,2.5411,-0.26,NaN,NaN,NaN,NaN,0.020357,0.020943,0.094421
2,SwingOption_20_c0.01_gamma1.5,0.01,1.5,12,21504,2.5026,2.4992,-0.14,NaN,NaN,NaN,NaN,0.020024,0.020816,0.089625
3,SwingOption_20_c0.01_gamma2,0.01,2.0,12,26624,2.4402,2.4472,0.29,0.0019,0.0067,0.0048,0.0086,0.020450,0.020639,0.079689
4,SwingOption_20_c0.01_gamma3,0.01,3.0,12,32768,2.2330,2.3164,3.73,0.0025,0.0831,0.0806,0.0857,0.020459,0.020003,0.113627


## Summary Statistics
Quick descriptive statistics for the main variables.

In [3]:
df.describe()

,c,gamma,Best Seed,Best Episode,LSM Price,RL Price,PctDiff,CI_95,MeanDiff,CI_Lower,CI_Upper,RL_95CI,LSM_95CI,PctDiff_95CI
count,25.000000,25.000000,25.000000,25.000000,25.000000,25.000000,25.000000,17.000000,17.000000,17.000000,17.000000,25.000000,25.000000,25.000000
mean,0.052800,1.700000,12.320000,22937.600000,1.862440,1.975164,9.031600,0.002359,0.142471,0.140118,0.144847,0.018718,0.018478,0.182266
std,0.042281,0.692219,0.690411,7770.489131,0.522837,0.421147,15.928566,0.000919,0.196502,0.195616,0.197386,0.001643,0.002123,0.179688
min,0.000000,1.000000,11.000000,5120.000000,0.969600,1.100900,-0.550000,0.001300,-0.010500,-0.012100,-0.008900,0.014937,0.014322,0.072369
25%,0.020000,1.000000,12.000000,16384.000000,1.389200,1.693200,-0.130000,0.001800,0.006700,0.004800,0.008600,0.017554,0.016694,0.089625
50%,0.040000,1.500000,12.000000,24576.000000,1.862200,1.975700,2.220000,0.002200,0.083100,0.080600,0.085700,0.019120,0.018772,0.113627
75%,0.080000,2.000000,13.000000,29696.000000,2.235000,2.316400,8.910000,0.002500,0.150900,0.148600,0.153200,0.020024,0.020063,0.143981
max,0.150000,3.000000,13.000000,32768.000000,2.661200,2.658500,63.720000,0.004900,0.658900,0.654000,0.663800,0.020965,0.021322,0.840343


## Pairwise Spearman Correlations
Spearman rank correlations capture monotonic relationships without assuming linearity.

In [4]:
spearman_corr = df[['c', 'gamma', 'PctDiff']].corr(method='spearman')
spearman_corr

,c,gamma,PctDiff
c,1.000000,-0.127515,0.378973
gamma,-0.127515,1.000000,0.777399
PctDiff,0.378973,0.777399,1.000000


## Scatter Visualization
Visualizing how `PctDiff` varies with `c` and `gamma`.

In [5]:
from itertools import cycle
from plotly.subplots import make_subplots
from plotly.colors import qualitative
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Use a renderer that loads MathJax (ensures LaTeX is rendered).
# Alternatives: "notebook_connected", "iframe_connected", "jupyterlab" (depending on your environment).
pio.renderers.default = "notebook_connected"

# --- Layout setup (tighter & balanced) ---
fig_plotly = make_subplots(
    rows=2, cols=2,
    subplot_titles=['PctDiff vs c', 'PctDiff vs gamma', '', ''],
    shared_yaxes=True,
    vertical_spacing=0.10,        # increased value for more space
    row_heights=[0.84, 0.16]
)

palette = qualitative.Plotly + qualitative.Safe

for idx, x_col in enumerate(['c', 'gamma'], start=1):
    # Color by the latent variable (gamma for left, c for right)
    color_var = 'gamma' if x_col == 'c' else 'c'
    unique_values = sorted(df[color_var].unique())
    palette_iter = cycle(palette)
    color_map = {val: next(palette_iter) for val in unique_values}

    # Scatter points by category with fixed colors
    for val in unique_values:
        mask = df[color_var] == val
        fig_plotly.add_trace(
            go.Scatter(
                x=df.loc[mask, x_col],
                y=df.loc[mask, 'PctDiff'],
                mode='markers',
                marker=dict(size=8, color=color_map[val]),
                hoverinfo='skip',
                showlegend=False
            ),
            row=1, col=idx
        )

    # Simple linear trend + ±1 SE band
    x_range = np.linspace(df[x_col].min(), df[x_col].max(), 200)
    pred_frame = pd.DataFrame({x_col: x_range})
    trend_fit = smf.ols(f'PctDiff ~ {x_col}', data=df).fit()
    pred_summary = trend_fit.get_prediction(pred_frame).summary_frame()
    mean = pred_summary['mean']
    se = pred_summary['mean_se']
    upper = mean + se
    lower = mean - se

    # ±1 SE shaded band
    fig_plotly.add_trace(
        go.Scatter(
            x=np.concatenate([x_range, x_range[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor='rgba(31, 119, 180, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            mode='lines',
            showlegend=False
        ),
        row=1, col=idx
    )

    # Mean line
    fig_plotly.add_trace(
        go.Scatter(
            x=x_range, y=mean, mode='lines',
            line=dict(color='rgba(31, 119, 180, 1)', width=2),
            name='Trend', showlegend=False
        ),
        row=1, col=idx
    )

    # Axis titles for top row
    fig_plotly.update_xaxes(title_text=x_col, row=1, col=idx)

    # Legend panels (compact horizontal layout)
    legend_x = list(range(len(unique_values)))
    legend_y = [0] * len(unique_values)
    legend_colors = [color_map[v] for v in unique_values]
    legend_labels = [format(v, '.3g') for v in unique_values]

    fig_plotly.add_trace(
        go.Scatter(
            x=legend_x, y=legend_y,
            mode='markers+text',
            marker=dict(color=legend_colors, size=12),
            text=legend_labels, textposition='bottom center',
            hoverinfo='skip', showlegend=False
        ),
        row=2, col=idx
    )

    # Title for legend panel (lowered so it doesn't push layout)
    fig_plotly.add_annotation(
        x=0.5, y=1.02, xref=f'x{idx+2} domain', yref='y domain',
        text=f"<b>{color_var}</b>", showarrow=False, font=dict(size=12),
        row=2, col=idx
    )

    # Hide axes in legend panels
    fig_plotly.update_xaxes(visible=False, row=2, col=idx)
    fig_plotly.update_yaxes(visible=False, row=2, col=idx)

# --- Shared y-axis label (proper LaTeX) ---
y_axis_title = r"$PctDiff = \frac{\mathrm{RL} - \mathrm{LSM}}{|\mathrm{LSM}|}$"
fig_plotly.update_yaxes(title_text=y_axis_title, row=1, col=1, title_standoff=14, automargin=True)

# --- Global layout ---
fig_plotly.update_layout(
    title=dict(
        text='RL vs LSM Pricing Gap under Convex Costs',
        x=0.5, xanchor='center',
        y=0.98, yanchor='top',
        pad=dict(t=2, b=0)
    ),
    template='plotly_white',
    height=600, width=980,
    margin=dict(l=80, r=30, t=70, b=50),
    font=dict(size=12)
)

# Improve axis-label spacing
fig_plotly.update_xaxes(title_standoff=10, row=1, col=1)
fig_plotly.update_xaxes(title_standoff=10, row=1, col=2)
fig_plotly.update_yaxes(title_standoff=14, row=1, col=1)

# Show with renderer that loads MathJax
fig_plotly.show(renderer="jupyterlab")

## Robust Linear Model
Fit an ordinary least squares model with heteroskedasticity-robust standard errors to test whether `c`, `gamma`, and their interaction explain `PctDiff`.

In [6]:
model = smf.ols('PctDiff ~ c + gamma + c:gamma', data=df).fit()
robust_res = model.get_robustcov_results(cov_type='HC3')
robust_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                PctDiff   R-squared:                       0.771
Model:                            OLS   Adj. R-squared:                  0.739
Method:                 Least Squares   F-statistic:                     7.175
Date:                Sat, 07 Mar 2026   Prob (F-statistic):            0.00170
Time:                        21:12:33   Log-Likelihood:                -85.730
No. Observations:                  25   AIC:                             179.5
Df Residuals:                      21   BIC:                             184.3
Df Model:                           3                                         
Covariance Type:                  HC3                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.0380      7.541     -0.801      0.432     -21.721       9.645
c           -417.0817    217.628     -1.916      0.069    -869.663      35.500
gamma          2.8305      4.609      0.614      0.546      -6.755      12.416
c:gamma      387.0462    139.167      2.781      0.011      97.632     676.460
==============================================================================
Omnibus:                       11.999   Durbin-Watson:                   2.296
Prob(Omnibus):                  0.002   Jarque-Bera (JB):               10.788
Skew:                           1.273   Prob(JB):                      0.00454
Kurtosis:                       4.968   Cond. No.                         191.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
"""

## Effect Grid
Predicted percent difference across the observed grid of `c` and `gamma`.

In [7]:
grid = df[['c', 'gamma']].drop_duplicates().sort_values(['c', 'gamma']).reset_index(drop=True)
grid['pred_pct_diff'] = robust_res.predict(grid)
grid

,c,gamma,pred_pct_diff
0,0.00,1.0,-3.207499
1,0.01,1.0,-3.507854
2,0.01,1.5,-0.157368
3,0.01,2.0,3.193118
4,0.01,3.0,9.894090
5,0.02,1.0,-3.808208
6,0.02,1.5,1.477509
7,0.02,2.0,6.763226
8,0.02,3.0,17.334660
9,0.04,1.0,-4.408917


## Interpretation
- The regression provides coefficient tests for `c`, `gamma`, and their interaction under robust standard errors.
- Combine the coefficient p-values with the Spearman correlations to assess whether higher convex costs correspond to larger RL advantages.
- Inspect the predicted grid to pinpoint regimes where RL diverges most from LSM.

In [8]:
print("Robust Regression Coefficients (PctDiff ~ c + gamma + c:gamma):\n")
for name, value in zip(robust_res.model.exog_names, robust_res.params):
    if name == "Intercept":
        desc = "Baseline PctDiff when c and gamma are zero"
    elif name == "c":
        desc = "Effect of c (convex cost parameter) on PctDiff"
    elif name == "gamma":
        desc = "Effect of gamma (convexity exponent) on PctDiff"
    elif name == "c:gamma":
        desc = "Interaction effect: how c's effect changes with gamma"
    else:
        desc = ""
    print(f"{name:10}: {value:10.4f}   # {desc}")

Robust Regression Coefficients (PctDiff ~ c + gamma + c:gamma):

Intercept :    -6.0380   # Baseline PctDiff when c and gamma are zero
c         :  -417.0817   # Effect of c (convex cost parameter) on PctDiff
gamma     :     2.8305   # Effect of gamma (convexity exponent) on PctDiff
c:gamma   :   387.0462   # Interaction effect: how c's effect changes with gamma


In [9]:
robust_res.pvalues

array([0.43229988, 0.06901261, 0.54573621, 0.0111921 ])

In [10]:
pctdiff_pivot = (
    df.pivot_table(index='c', columns='gamma', values='PctDiff', aggfunc='mean')
      .sort_index()
      .reindex(sorted(df['gamma'].unique()), axis=1)
)

abs_max = np.nanmax(np.abs(pctdiff_pivot.values))
styled_pivot = (
    pctdiff_pivot.round(3)
    .style.format(precision=3)
    .background_gradient(cmap='coolwarm', axis=None, vmin=-abs_max, vmax=abs_max)
)
ctdiff_pivot = (
    df.pivot_table(index='c', columns='gamma', values='PctDiff', aggfunc='mean')
      .sort_index()
      .reindex(sorted(df['gamma'].unique()), axis=1)
)

def border_color(val):
    if pd.isna(val):
        return ''
    color = 'green' if val > 0 else 'red'
    return f'border: 2px solid {color};'

styled_pivot = (
    pctdiff_pivot.round(3)
    .style.format(precision=3)
    .background_gradient(cmap='coolwarm', axis=None, vmin=-abs_max, vmax=abs_max)
    .applymap(border_color)
)
styled_pivot

/var/folders/57/8q11hb450rz9z_rwvfbzpkmm0000gn/T/ipykernel_74193/110698141.py:29: FutureWarning:

Styler.applymap has been deprecated. Use Styler.map instead.



gamma,1.000000,1.500000,2.000000,3.000000
c,,,,
0.000000,-0.100,nan,nan,nan
0.010000,-0.260,-0.140,0.290,3.730
0.020000,-0.340,0.350,1.530,12.300
0.040000,-0.240,1.330,5.980,42.570
0.050000,-0.190,2.220,8.910,63.720
0.080000,-0.550,4.020,25.360,nan
0.100000,-0.130,6.950,32.710,nan
0.150000,2.230,13.540,nan,nan
